In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# Load datasets
data_path = 'data/raw/'
transactions = pd.read_csv(os.path.join(data_path, 'transactions_train.csv'))
articles = pd.read_csv(os.path.join(data_path, 'articles.csv'))
customers = pd.read_csv(os.path.join(data_path, 'customers.csv'))

# Convert date
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
print(f"Transactions: {transactions.shape}")

Transactions: (31788324, 5)


In [2]:
print("=" * 70)
print("BASELINE MODEL: MOST POPULAR ITEMS")
print("=" * 70)

# 1. Get most popular articles overall
article_popularity = transactions.groupby('article_id').size().reset_index(name='purchase_count')
article_popularity = article_popularity.sort_values('purchase_count', ascending=False)

# Top 12 articles (Kaggle requires top-12 recommendations)
top_12_articles = article_popularity.head(12)['article_id'].tolist()

print(f"\n1. TOP 12 MOST POPULAR ARTICLES (BASELINE):")
for idx, article_id in enumerate(top_12_articles, 1):
    count = article_popularity[article_popularity['article_id'] == article_id]['purchase_count'].values[0]
    article_name = articles[articles['article_id'] == article_id]['prod_name'].values[0]
    print(f"   {idx:>2}. Article {article_id}: {count:>6,} purchases | {article_name[:50]}")

# 2. Create submission format
print(f"\n2. SUBMISSION FORMAT (Kaggle requirement):")
print(f"   Each customer → Top 12 recommendations (same for all customers)")
print(f"   Format: customer_id | article_id_1 article_id_2 ... article_id_12")

# Get unique customers
unique_customers = transactions['customer_id'].unique()
print(f"\n   Total customers for prediction: {len(unique_customers):,}")

# Create baseline submission
baseline_submission = pd.DataFrame()
baseline_submission['customer_id'] = unique_customers
baseline_submission['prediction'] = baseline_submission.apply(
    lambda row: ' '.join(map(str, top_12_articles)), 
    axis=1
)

print(f"\n   Submission shape: {baseline_submission.shape}")
print(f"\n   Sample predictions:")
print(baseline_submission.head())

# 3. Save submission
submission_path = 'submissions/baseline_submission.csv'
os.makedirs('submissions', exist_ok=True)
baseline_submission.to_csv(submission_path, index=False)
print(f"\n✅ Baseline submission saved: {submission_path}")
print(f"   File size: {os.path.getsize(submission_path) / 1024:.1f} KB")

BASELINE MODEL: MOST POPULAR ITEMS

1. TOP 12 MOST POPULAR ARTICLES (BASELINE):
    1. Article 706016001: 50,287 purchases | Jade HW Skinny Denim TRS
    2. Article 706016002: 35,043 purchases | Jade HW Skinny Denim TRS
    3. Article 372860001: 31,718 purchases | 7p Basic Shaftless
    4. Article 610776002: 30,199 purchases | Tilly (1)
    5. Article 759871002: 26,329 purchases | Tilda tank
    6. Article 464297007: 25,025 purchases | Greta Thong Mynta Low 3p
    7. Article 372860002: 24,458 purchases | 7p Basic Shaftless
    8. Article 610776001: 22,451 purchases | Tilly (1)
    9. Article 399223001: 22,236 purchases | Curvy Jeggings HW Ankle
   10. Article 706016003: 21,241 purchases | Jade HW Skinny Denim TRS
   11. Article 720125001: 21,063 purchases | SUPREME RW tights
   12. Article 156231001: 21,013 purchases | Box 4p Tights

2. SUBMISSION FORMAT (Kaggle requirement):
   Each customer → Top 12 recommendations (same for all customers)
   Format: customer_id | article_id_1 articl

In [3]:
print(f"\n5. COVERAGE ANALYSIS:")
# Customers who bought from top-12
top_12_set = set(top_12_articles)
customers_with_top12 = transactions[transactions['article_id'].isin(top_12_set)]['customer_id'].unique()
coverage = len(customers_with_top12) / len(unique_customers) * 100

print(f"   Customers who bought from top-12: {len(customers_with_top12):,} ({coverage:.1f}%)")
print(f"   → Only {coverage:.1f}% will have perfect score with this baseline")
print(f"   → {100-coverage:.1f}% will have 0 score (no match)")

# 4. Top-12 composition
print(f"\n6. TOP-12 COMPOSITION:")
top_12_data = article_popularity.head(12).merge(articles[['article_id', 'product_type_name', 'colour_group_name']], 
                                                 on='article_id', how='left')
print(f"\n   By Category:")
cat_dist = top_12_data['product_type_name'].value_counts()
for cat, count in cat_dist.items():
    print(f"   - {cat}: {count}")

print(f"\n   By Color:")
color_dist = top_12_data['colour_group_name'].value_counts()
for color, count in color_dist.items():
    print(f"   - {color}: {count}")

print(f"\n7. INSIGHTS:")
print(f"   ✓ Denim dominates (Jade variants = 3/12)")
print(f"   ✓ Basics popular (Shaftless, Tilda, Tights)")
print(f"   ✓ Black & White preferred colors")
print(f"   → Next model should respect these patterns")


5. COVERAGE ANALYSIS:
   Customers who bought from top-12: 180,865 (13.3%)
   → Only 13.3% will have perfect score with this baseline
   → 86.7% will have 0 score (no match)

6. TOP-12 COMPOSITION:

   By Category:
   - Trousers: 4
   - Socks: 2
   - T-shirt: 2
   - Vest top: 1
   - Underwear bottom: 1
   - Leggings/Tights: 1
   - Underwear Tights: 1

   By Color:
   - Black: 8
   - White: 2
   - Light Blue: 1
   - Dark Blue: 1

7. INSIGHTS:
   ✓ Denim dominates (Jade variants = 3/12)
   ✓ Basics popular (Shaftless, Tilda, Tights)
   ✓ Black & White preferred colors
   → Next model should respect these patterns


In [4]:
print("\n" + "=" * 70)
print("IMPROVED BASELINE: CUSTOMER-SPECIFIC TOP-12")
print("=" * 70)

# Strategy: For each customer, recommend top items from THEIR preferred categories
print(f"\n1. STRATEGY: Category-Based Personalization")

# Get customer's preferred categories
customer_categories = transactions.merge(articles[['article_id', 'product_type_name']], 
                                        on='article_id').groupby('customer_id')['product_type_name'].value_counts().reset_index(name='count')

# For each customer, get top articles from their top category
def get_personalized_recommendations(customer_id, n_recommendations=12):
    # Get customer's top category
    customer_top_cat = customer_categories[customer_categories['customer_id'] == customer_id]
    if len(customer_top_cat) == 0:
        # Fallback to global top-12
        return top_12_articles
    
    top_cat = customer_top_cat.iloc[0]['product_type_name']
    
    # Get top articles from this category
    cat_articles = article_popularity.merge(articles[['article_id', 'product_type_name']], 
                                            on='article_id')
    cat_articles = cat_articles[cat_articles['product_type_name'] == top_cat]
    
    if len(cat_articles) < n_recommendations:
        # Not enough in category, fill with global top
        recommendations = cat_articles['article_id'].head(n_recommendations).tolist()
        remaining = n_recommendations - len(recommendations)
        global_remaining = [a for a in top_12_articles if a not in recommendations]
        recommendations.extend(global_remaining[:remaining])
    else:
        recommendations = cat_articles['article_id'].head(n_recommendations).tolist()
    
    return recommendations

# Create improved submission
print(f"\n2. GENERATING PERSONALIZED RECOMMENDATIONS...")
improved_submission = pd.DataFrame()
improved_submission['customer_id'] = unique_customers[:1000]  # Sample for speed
improved_submission['prediction'] = improved_submission['customer_id'].apply(
    lambda cid: ' '.join(map(str, get_personalized_recommendations(cid)))
)

print(f"   Sample personalized predictions:")
print(improved_submission.head(10))

print(f"\n✅ Personalized baseline ready")
print(f"   Expected improvement: Better category matching")


IMPROVED BASELINE: CUSTOMER-SPECIFIC TOP-12

1. STRATEGY: Category-Based Personalization

2. GENERATING PERSONALIZED RECOMMENDATIONS...
   Sample personalized predictions:
                                         customer_id  \
0  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
1  00007d2de826758b65a93dd24ce629ed66842531df6699...   
2  00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4...   
3  0008968c0d451dbc5a9968da03196fe20051965edde741...   
4  000aa7f0dc06cd7174389e76c9e132a67860c5f65f9706...   
5  001127bffdda108579e6cb16080440e89bf1250a776c6e...   
6  001ea4e9c54f7e9c88811260d954edc059d596147e1cf8...   
7  001fd23db1109a94bba1319bb73df0b479059027c182da...   
8  0021da829b898f82269fc51feded4eac2129058ee95bd7...   
9  00228762ecff5b8d1ea6a2e52b96dafa198febddbc3bf3...   

                                          prediction  
0  673677002 537116001 685813001 591334003 677930...  
1  723469001 564786001 579302001 736530007 253448...  
2  351484002 688537004 599580017 688537011 60